# Задание 3
Что нужно сделать
* Обучите RNN, используя все изученные подходы. 
* Для стабильности обучения нужно использовать:
  * gradient clipping;
  * xavier инициализацию весов;
  * регуляризацию (dropout в модели и weight_decay в оптимизаторе);
  * нормализацию;
  * mean pooling, изученный в этом уроке, для улучшения качества.
* Для инициализации весов используйте nn.init.xavier_uniform_.
* Чтобы посчитать маску для mean pooling, нужно использовать unsqueeze и expand_as для маски, которая приходит на вход метода forward.
* Нормализация должна использоваться после применения RNN.
* dropout должен считаться после расчёта mean pooling.

**NB!:** В коде ниже переменные texts и labels переиспользуются из предыдущих заданий. Код создания даталоадеров для тренировочного и валидационного датасета уже написан — они сохранены в переменные train_loader и val_loader, а токенизатор — в переменную tokenizer. Вам нужно дописать код модели и обучить нейросеть.

In [5]:
import pandas as pd
from transformers import BertTokenizerFast
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [6]:

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [7]:
class AmazonRNNDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, padding='max_length', truncation=True, max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels)


    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label': self.labels[idx]
        }

In [8]:


ddf = pd.read_csv('../data/amazon_dataset.csv', sep=';')
texts, labels = list(ddf['content']), list(ddf['label'])

X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42)

train_ds = AmazonRNNDataset(X_train, y_train, tokenizer)
val_ds = AmazonRNNDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

In [9]:
class MeanPoolingRNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=300, hidden_dim=256, output_dim=2, pad_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, output_dim)

        self.init_weights()


    def init_weights(self):
        nn.init.xavier_uniform_(self.fc.weight)
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)


    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        rnn_out, _ = self.rnn(x)
        rnn_out_normed = self.norm(rnn_out)

        # mean pooling по attention_mask
        mask = attention_mask.unsqueeze(2).expand_as(rnn_out_normed)
        masked_out = rnn_out_normed * mask
        summed = masked_out.sum(dim=1)
        lengths = attention_mask.sum(dim=1).unsqueeze(1)
        mean_pooled = summed / lengths
        out = self.dropout(mean_pooled)
        logits = self.fc(out)

        return logits


In [10]:
# создание модели
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MeanPoolingRNN(vocab_size=tokenizer.vocab_size, pad_idx=tokenizer.pad_token_id).to(device)

# создание оптимизатора и функции потерь
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss()


In [11]:
# код обучения одной эпохи
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)


        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [12]:
# код подсчёта accuracy на валидации
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['label']
            logits = model(ids, mask)
            preds += torch.argmax(logits, dim=1).cpu().tolist()
            trues += labels.tolist()
    return accuracy_score(trues, preds)

In [13]:
# обучение
for epoch in range(3):
    loss = train_epoch(model, train_loader)
    acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Accuracy = {acc:.4f}")

  0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 1: Loss = 0.3519, Accuracy = 0.8663


  0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2: Loss = 0.2859, Accuracy = 0.8784


  0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 3: Loss = 0.2626, Accuracy = 0.8716
